In [1]:
import pandas as pd
import numpy as np
import utils
import os
from pathlib import Path

In [ ]:
def getPlayersMatch(df_events):
  teams_player = df_events.teamId.unique()
  players = {}
  for id, player in zip(df_events.playerId, df_events.playerName):
    if not np.isnan(id):
      players[id] = player

  return players

In [4]:
def translateRelatedEvent(satisfiedEvents):
  file = 'Dataset/WhoScored/event_metadata.json'
  event_metadata = utils.readJson(file)
  event_metadata = {value : key for key,value in event_metadata.items()}
  return [event_metadata[x] for x in satisfiedEvents]

def misureDistance(x1, y1, x2, y2):
  return np.sqrt((x1-x2)**2 + (y1-y2)**2)

def traduciEvento(events_dict,event):
  #events_dict = readJson( f'{GDRIVE_THESIS_DIR}/action_translations.json')
  events = events_dict['events']
  related_events = events_dict['relatedEvents']
  current_event = event['displayName']
  translated_event = events[current_event]
  #current_related_events = translateRelatedEvent(satisfiedEvents)
  #translated_related_events = [related_events[x] for x in current_related_events if related_events[x] != '']
  #str_related_events = f" ({', '.join(translated_related_events).strip(', ')})" if len(translated_related_events) > 0 else ""
  return translated_event

def traduciEventiSupplementari(events_dict, satisfiedEvents, qualifiers):
  qualifiers_dict = events_dict['qualifiers']
  related_events = events_dict['relatedEvents']
  current_related_events = translateRelatedEvent(satisfiedEvents)
  qualifierNames = [x['type']['displayName'] for x in qualifiers]
  translated_qualifiers = [qualifiers_dict[x] for x in qualifierNames if qualifiers_dict[x] != '']
  translated_related_events = [related_events[x] for x in current_related_events if related_events[x] != ''] + translated_qualifiers
  str_related_events = f" ({', '.join(translated_related_events).strip(', ')})" if len(translated_related_events) > 0 else ""

  return str_related_events


def componiFrase(row, events_dict):
    #esito = 'con successo' if row['outcomeType']['value'] == 1 else 'fallendo'
    start_position = detectFieldBin(row['x'], row['y'])
    if start_position != '':
      start_position = f" da {start_position}"
    else:
      start_position = ''

    if np.isnan(row['endX']) or np.isnan(row['endY']):
      end_position = ''
    else:
      end_position = f" a {detectFieldBin(row['endX'], row['endY'])}"
    '''
    if pd.isna(row['playerName']):
      player=''
    else:
      player = row['playerName']

    if pd.isna(row['teamName']):
      team =''
    else:
      team = f"({row['teamName']})"
  '''
    #frase = f"{player} {team} {traduciEvento(events_dict, row['type'])} {start_position}{end_position} {traduciEventiSupplementari(events_dict, row['satisfiedEventsTypes'], row['qualifiers'])}"
    frase = f"{traduciEvento(events_dict, row['type'])} {start_position}{end_position} {traduciEventiSupplementari(events_dict, row['satisfiedEventsTypes'], row['qualifiers'])}"
    return frase.strip()

def detectFieldBin(x , y, n_binx: int = 5, n_biny: int = 5):
    if x == 0 and y == 0:
      return ''

    x_text = {1: 'difesa', 2: 'trequarti difensiva', 3: 'centrocampo', 4: 'trequarti offensiva', 5: 'attacco'}
    y_text = {1: 'fascia destra', 2: 'centro destra', 3: 'centrale', 4: 'centro sinistra', 5: 'fascia sinistra'}
    field_length_x = 100
    field_length_y = 100
    bin_x_width, bin_y_width = np.ceil(field_length_x / n_binx), np.ceil(field_length_y / n_biny)
    bin_x = int((x - 1) / bin_x_width) + 1
    bin_y = int((y - 1) / bin_y_width) + 1
    return f"{x_text[bin_x]}, {y_text[bin_y]}"


In [ ]:
def componiTextPlayerMatch(player_match: dict) -> str:
    text = ''
    i = 0
    for id, action in player_match['text'].items():
        if i > 0:
            if int(id) == i+1:
                text = text +', ' + action
            else:
                text = text +'. ' + action
        else:
            text = action
        i = int(id)

    return text

In [ ]:
src_dir = os.path.join('Dataset', 'WhoScored')
events_dict = utils.readJson( f'{src_dir}/action_translations.json')
new_related_events = set()
new_qualifiers = set()
new_events = set()


src_dir_path = Path(src_dir)
src_dir_league = [d.name for d in src_dir_path.iterdir() if d.is_dir() and '-' in d.name]
for league_dir in src_dir_league:
    league_dir_abs = os.path.join(src_dir, league_dir)
    tgt_league_dir_abs = league_dir_abs.replace('WhoScored','Events2Text')
    #os.makedirs(tgt_league_dir_abs, exist_ok=True)
    league_dir_path = Path(league_dir_abs)
    src_dir_league_season = [d.name for d in league_dir_path.iterdir() if d.is_dir() and '-' in d.name]
    for season in src_dir_league_season:
        src_dir_full = os.path.join(league_dir_abs, season)
        tgt_dir_full = os.path.join(tgt_league_dir_abs,season)
        #os.makedirs(tgt_dir_full, exist_ok=True)
        input_files = os.listdir(src_dir_full)
        for match in input_files:
            match_events = []
            file = os.path.join(src_dir_full, match)
            df_events = pd.read_json(file)

            for idx, row in df_events.iterrows():
                '''
                qualifiers = [x['type']['displayName'] for x in row['qualifiers']]
                event = row['type']['displayName']
                if event not in events_dict['events'].keys():
                    new_events.add(event)
                for q in qualifiers:
                    if q not in events_dict['qualifiers'].keys():
                        new_qualifiers.add(q)
                related_events = translateRelatedEvent(row['satisfiedEventsTypes'])'
                for rev in related_events:
                    if rev not in events_dict['relatedEvents'].keys():
                        new_related_events.add(rev)
                '''
                match_events.append({'playerId': row['playerId'], 'playerName': row['playerName'],'teamId': row['teamId'], 'team': row['teamName'], 'text': componiFrase(row, events_dict)})
                
            utils.writeJson(match_events, os.path.join(tgt_dir_full, match))


In [56]:
def getPlayersMatch(df_events):
    teams = {}
    try:
        for id, name in zip(df_events.teamId, df_events.teamName):
            if not pd.isna(name):
                teams[id] = name
    except:
        for id, name in zip(df_events.teamId, df_events.team):
            if not pd.isna(name):
                teams[id] = name

    teams_list = {x: { 'name': teams[x], 'players': {}} for x in teams.keys()}

    for id, row in df_events.iterrows():
        teamId = row['teamId']
        id = row['playerId']
        if not np.isnan(id):
            teams_list[teamId]['players'][id] = row['playerName']

    return teams_list

def aggiungiPlayersFromMatch(teams_list_all, teams_list_match):
    for team in teams_list_match.keys():
        if team not in teams_list_all:
            teams_list_all[team] = teams_list_match[team]
        else:
            teams_list_all[team]['players'] = teams_list_all[team]['players'] | teams_list_match[team]['players'] 


Numero di eventi totali: 20.846.939 <br>
Numero di partite totali: 13.331

In [52]:
src_dir = os.path.join('Dataset', 'WhoScored')
tgt_dir = os.path.join('Dataset', 'Events2Text')

src_dir_path = Path(src_dir)
src_dir_league = [d.name for d in src_dir_path.iterdir() if d.is_dir() and '-' in d.name]
for league_dir in src_dir_league:
    league_dir_abs = os.path.join(src_dir, league_dir)
    league_dir_path = Path(league_dir_abs)
    src_dir_league_season = [d.name for d in league_dir_path.iterdir() if d.is_dir() and '-' in d.name]
    for season in src_dir_league_season:
        src_dir_full = os.path.join(league_dir_abs, season)
        input_files = os.listdir(src_dir_full)
        teams_list = {}
        for match in input_files:
            file = os.path.join(src_dir_full, match)
            df_events = pd.read_json(file)
            match_players = getPlayersMatch(df_events)
            aggiungiPlayersFromMatch(teams_list, match_players)
        
        utils.writeJson(teams_list, os.path.join(tgt_dir, f'{league_dir}_{season}_teams.json'))
            

In [69]:
src_dir = os.path.join('Dataset', 'Events2Text')
out_dir = os.path.join(src_dir, 'PlayerDocs')
src_dir_path = Path(src_dir)
src_dir_league = [d.name for d in src_dir_path.iterdir() if d.is_dir() and '-' in d.name]
for league_dir in src_dir_league:
    league_dir_abs = os.path.join(src_dir, league_dir)
    league_dir_path = Path(league_dir_abs)
    src_dir_league_season = [d.name for d in league_dir_path.iterdir() if d.is_dir() and '-' in d.name]
    for season in src_dir_league_season:
        src_dir_full = os.path.join(league_dir_abs, season)
        input_files = os.listdir(src_dir_full)
        teams_list = {}
        for match in input_files:
            file = os.path.join(src_dir_full, match)
            df_events = pd.read_json(file)
            team_players = getPlayersMatch(df_events)
            for team in team_players.keys():
                ply = team_players[team]['players']
                for p in ply.keys():
                    df_p_events = df_events[df_events.playerId == p]
                    out_dir_p = os.path.join(out_dir, str(p))
                    os.makedirs(out_dir_p, exist_ok=True)
                    out_dir_p_season = os.path.join(out_dir_p, season)
                    os.makedirs(out_dir_p_season, exist_ok = True)
                    df_p_events.to_json(os.path.join(out_dir_p_season, match))
        


In [68]:
path = 'Dataset\\Events2Text\\PlayerDocs\\460260.0\\2023-2024\\2023-12-13_Man Utd_Bayern.json'
len(pd.read_json(path).head())

5

In [ ]:
src_dir = os.path.join('Dataset', 'WhoScored')

teams_list = {}
for match in os.listdir(src_dir):
    df_events = pd.read_json(os.path.join(src_dir, match))
    match_players = getPlayersMatch(df_events)
    aggiungiPlayersFromMatch(teams_list, match_players)

teams_list

In [38]:
teams_list.keys()

dict_keys([76, 272, 276, 2732, 73, 278, 269, 75, 77, 79, 84, 143, 300, 2889, 71, 80, 78, 72, 86, 87])

In [50]:
len(teams_list[80]['players'])

35

In [ ]:
#print("calcio d'inizio")
events_dict = utils.readJson( f'{src_dir}/action_translations.json')

new_related_events = set()
new_qualifiers = set()
new_events = set()
for match in os.listdir(src_dir_league_season):
    match_events = []
    file = os.path.join(src_dir_league_season, match)
    df_events = pd.read_json(file)

    for idx, row in df_events.iterrows():
        event = row['type']['displayName']
        events = events_dict['events']
        if event not in events.keys():
            new_events.add(event)
        match_events.append({'playerId': row['playerId'], 'playerName': row['playerName'], 'team': row['teamName'], 'text': componiFrase(row, events_dict)})

    #utils.writeJson(match_events, os.path.join(tgt_dir_league_season, match))
#print('fischio finale')
new_events

set()

In [26]:
new_related_events

{'assistCorner',
 'goalLeftFoot',
 'keeperClaimHighWon',
 'keeperMissed',
 'penaltyConceded',
 'penaltyScored',
 'penaltyWon',
 'punches',
 'redCard',
 'saveSixYardBox',
 'secondYellow',
 'voidYellowCard'}

In [27]:
new_qualifiers

{'HighClaim',
 'HighLeft',
 'KeeperMissed',
 'KeeperSaveInSixYard',
 'Penalty',
 'SecondYellow',
 'SmallBoxRight',
 'VoidYellowCard'}

In [ ]:
events_dict = {x['value']: x['displayName'] for x in df_events.type}
events_dict

{32: 'Start',
 1: 'Pass',
 49: 'BallRecovery',
 12: 'Clearance',
 61: 'BallTouch',
 45: 'Challenge',
 3: 'TakeOn',
 7: 'Tackle',
 15: 'SavedShot',
 10: 'Save',
 52: 'KeeperPickup',
 50: 'Dispossessed',
 74: 'BlockedPass',
 6: 'CornerAwarded',
 44: 'Aerial',
 8: 'Interception',
 13: 'MissedShots',
 16: 'Goal',
 4: 'Foul',
 54: 'Smother',
 59: 'KeeperSweeper',
 17: 'Card',
 14: 'ShotOnPost',
 10000: 'OffsideGiven',
 2: 'OffsidePass',
 55: 'OffsideProvoked',
 30: 'End',
 18: 'SubstitutionOff',
 19: 'SubstitutionOn',
 40: 'FormationChange',
 56: 'ShieldBallOpp',
 34: 'FormationSet'}

<h1>Salvare tutto il dataset in un unico json</h1>

In [ ]:
src_dir = 'Dataset\\Events2Text\\PlayerDocs'
tgt_dir = 'Dataset\\Events2Text'
entire_dataset = []
j=0
for p in os.listdir(src_dir):
    src_dir_p = os.path.join(src_dir, p)
    for s in os.listdir(src_dir_p):
        src_dir_p_s = os.path.join(src_dir_p, s)
        for match in os.listdir(src_dir_p_s):
            player_match = utils.readJson(os.path.join(src_dir_p_s, match))
            teamId = list(player_match['teamId'].values())[0]
            team = list(player_match['team'].values())[0]
            playerId = list(player_match['playerId'].values())[0]
            playerName = list(player_match['playerName'].values())[0]
            record = dict(season=s, playerId=playerId, playerName=playerName, teamId=teamId, teamName=team)
            record['text'] = componiTextPlayerMatch(player_match)
            record['match'] = match
            entire_dataset.append(record)

utils.writeJson(entire_dataset, os.path.join(tgt_dir, 'player2vec_dataset.json'))